# Measure Twice - SFT (Qwen3-1.7B, Unsloth 4-bit LoRA)
Teaches the tool-call format and the "fewest calls" behaviour from oracle episodes (`data/sft.jsonl`).
Expected runtime on one T4: ~30-60 min for a few thousand episodes.

**Setup.** Kaggle: GPU T4 x2, Internet ON. Attach the repo as a Kaggle Dataset (default mount below) or uncomment the `git clone` line.
The repo is copied to `/kaggle/working/mt` because `/kaggle/input` is read-only (and `gen_sft.py` writes into `data/`).

In [ ]:
import os, sys, shutil
os.environ["CUDA_VISIBLE_DEVICES"] = "0"   # Unsloth trains on one GPU; the 2nd T4 stays idle
os.environ["TOKENIZERS_PARALLELISM"] = "false"

REPO = "/kaggle/input/measure-twice"         # Kaggle Dataset mount of the repo folder
# !git clone https://github.com/<you>/measure-twice /kaggle/working/mt-src && REPO = "/kaggle/working/mt-src"
WORK = "/kaggle/working/mt"
shutil.copytree(REPO, WORK, dirs_exist_ok=True)
os.chdir(WORK)
sys.path.insert(0, WORK)
print(os.listdir(WORK))

In [ ]:
!pip install -q --upgrade unsloth unsloth_zoo
# if the import below fails after an Unsloth release, pin to the last good versions (pip freeze from a working run)

**Choices for T4.**
- T4 has no bf16, so the model loads in fp16 (`dtype=None` lets Unsloth pick) and the trainer uses `fp16=True`.
- `fast_inference=False`: Unsloth's vLLM path is unreliable on T4 (compute capability 7.5, small VRAM shared with training). Plain HF `generate` is slower but always works, and `rollout.hf_generator` uses it anyway.
- **Loss masking:** each episode is split into one example per assistant turn, rendered with
  `apply_chat_template(messages[:i], add_generation_prompt=True, enable_thinking=False)` - byte-identical to what `rollout.hf_generator` feeds the model at inference.
  Only the assistant completion (+ `<|im_end|>`) carries loss. This sidesteps a Qwen3-template quirk: when rendering a whole conversation it only puts the empty
  `<think>\n\n</think>` block on the *last* assistant turn, while inference always pre-fills it, so whole-conversation training with
  `train_on_responses_only` would train on a prompt format the model never sees at inference. Masking is done by hand and fed to a plain HF `Trainer`, so nothing depends on SFTTrainer/Unsloth masking internals.

In [ ]:
from unsloth import FastLanguageModel   # import unsloth before trl/transformers so its patches apply
import torch

BASE = "Qwen/Qwen3-1.7B"
MAX_SEQ = 2048
LORA = dict(r=16, lora_alpha=32, lora_dropout=0.0, bias="none",
            target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
            use_gradient_checkpointing="unsloth", random_state=0)

model, tokenizer = FastLanguageModel.from_pretrained(
    BASE, max_seq_length=MAX_SEQ, load_in_4bit=True, dtype=None,  # dtype=None -> fp16 on T4 (no bf16)
    fast_inference=False)                                          # no vLLM on T4, see note above
model = FastLanguageModel.get_peft_model(model, **LORA)

In [ ]:
import json, subprocess
from pathlib import Path

if not Path("data/sft.jsonl").exists():
    subprocess.run([sys.executable, "gen_sft.py"], check=True)
episodes = [json.loads(l)["messages"] for l in open("data/sft.jsonl")]
print(len(episodes), "episodes")

def step_examples(msgs):
    for i, m in enumerate(msgs):
        if m["role"] != "assistant":
            continue
        prompt = tokenizer.apply_chat_template(msgs[:i], tokenize=False, add_generation_prompt=True, enable_thinking=False)
        p = tokenizer(prompt, add_special_tokens=False)["input_ids"]
        c = tokenizer(m["content"] + "<|im_end|>", add_special_tokens=False)["input_ids"]
        ids = (p + c)[:MAX_SEQ]
        labels = ([-100] * len(p) + c)[:MAX_SEQ]
        if any(l != -100 for l in labels):
            yield {"input_ids": ids, "attention_mask": [1] * len(ids), "labels": labels}

from datasets import Dataset
rows = [ex for msgs in episodes for ex in step_examples(msgs)]
ds = Dataset.from_list(rows).shuffle(seed=0)
lens = sorted(len(r["input_ids"]) for r in rows)
print(len(rows), "step examples; token len p50/p95/max =", lens[len(lens)//2], lens[int(len(lens)*.95)], lens[-1])

# sanity: the exact text that carries loss in one example
print(repr(tokenizer.decode([t for t in ds[0]["labels"] if t != -100])))

In [ ]:
from transformers import Trainer, TrainingArguments, DataCollatorForSeq2Seq
from unsloth import is_bfloat16_supported

args = TrainingArguments(
    output_dir="/kaggle/working/sft_ckpt",
    num_train_epochs=1,
    per_device_train_batch_size=8,
    gradient_accumulation_steps=2,      # effective batch 16 step-examples
    learning_rate=2e-4,                 # standard for LoRA SFT
    lr_scheduler_type="cosine",
    warmup_ratio=0.03,
    weight_decay=0.0,
    fp16=not is_bfloat16_supported(),
    bf16=is_bfloat16_supported(),
    optim="adamw_8bit",
    logging_steps=10,
    save_steps=200, save_total_limit=2,
    group_by_length=True,               # less padding
    report_to="none",
    remove_unused_columns=False,
    seed=0,
)
trainer = Trainer(model=model, args=args, train_dataset=ds,
                  data_collator=DataCollatorForSeq2Seq(tokenizer, padding=True, label_pad_token_id=-100))
trainer.train()

In [ ]:
SFT_LORA = "/kaggle/working/sft_lora"
model.save_pretrained(SFT_LORA)
tokenizer.save_pretrained(SFT_LORA)
print(os.listdir(SFT_LORA))

## Quick val eval (30 tasks)

In [ ]:
import statistics as st
import rollout
from sandbox import tasks
from unsloth import FastLanguageModel

def evaluate(model, tokenizer, n=30, tag=""):
    FastLanguageModel.for_inference(model)
    gen = rollout.hf_generator(model, tokenizer, max_new_tokens=256)
    eps = [rollout.run_episode(t, gen) for t in tasks.generate("val", n)]
    keys = ["success", "n_calls", "n_redundant", "n_unsafe", "n_invalid"]
    res = {k: st.mean(float(e[k]) for e in eps) for k in keys}
    print(tag, {k: round(v, 3) for k, v in res.items()})
    return res, eps

In [ ]:
sft_res, sft_eps = evaluate(model, tokenizer, 30, "SFT")
# Optional baseline (same weights, adapter off):
# with model.disable_adapter(): base_res, _ = evaluate(model, tokenizer, 30, "BASE")

In [ ]:
# Optional: push the adapter to the HF Hub. Add a Kaggle Secret named HF_TOKEN (Add-ons -> Secrets); never hardcode tokens.
PUSH = False
HF_REPO = "<your-username>/measure-twice-sft"
if PUSH:
    from kaggle_secrets import UserSecretsClient
    tok = UserSecretsClient().get_secret("HF_TOKEN")
    model.push_to_hub(HF_REPO, token=tok, private=True)
    tokenizer.push_to_hub(HF_REPO, token=tok, private=True)